# Phase 3: Text CNN + BiGRU (v2 splits)

Same architecture and hyperparameters as the v1 Phase 3 run, retrained on the **v2 splits**
from `notebooks/datagen/email_phishing_datagen_v2.ipynb`.

The v1 splits merged an anonymized duplicate of TREC-05/06/07 that defeated near-duplicate
detection, leaving 46.5% of the v1 test set with a subject-twin in v1 train. v2 removes that
source. Holding the model fixed and changing only the data isolates the effect of the fix, so
the v1 and v2 metrics are directly comparable.

Expect v2 numbers to be **lower** than v1's published numbers and close to the v1 model's
measured clean-slice performance (F1 0.9812, FPR 2.23%). That is the correction, not a
regression: v1's published numbers were inflated by the contamination.

The v1 artifacts under `models/phase3-cnn-bigru-a1e06b36c0af0662/` and the v1 splits under
`data/source-clean/` are left untouched.


In [1]:
import gc
import hashlib
import json
import os
import random
import re
from collections import Counter
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, brier_score_loss, confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    try:
        torch.cuda.set_per_process_memory_fraction(0.55, 0)
    except RuntimeError as exc:
        print(f'CUDA memory cap was not applied: {exc}')

DATASET_VERSION = 'v2'
# Resolve the project root from the environment so this notebook runs both in the
# /workspace notebook container and directly on the host.
_candidates = [
    Path(p) for p in (
        os.environ.get('EMAIL_NN_DIR'),
        '/workspace/training/email-classifier-nn',
        '/home/spark/projects/training/email-classifier-nn',
    ) if p
]
PROJECT_DIR = next((p for p in _candidates if p.exists()), None)
assert PROJECT_DIR is not None, f'project root not found; tried {_candidates}'
DATA_DIR = PROJECT_DIR / 'data' / f'source-clean-{DATASET_VERSION}'
MANIFEST_PATH = DATA_DIR / 'manifest.json'
assert DATA_DIR.exists(), f'v2 splits missing: {DATA_DIR} (run the v2 datagen notebook first)'
print({'project': str(PROJECT_DIR), 'data': str(DATA_DIR), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(),
       'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})


{'project': '/home/spark/projects/training/email-classifier-nn', 'data': '/home/spark/projects/training/email-classifier-nn/data/source-clean-v2', 'torch': '2.11.0+cu130', 'cuda': True, 'device': 'NVIDIA GB10'}


## 1. Load v2 Splits

Only `train.parquet` fits vocabulary and model parameters. Validation controls early stopping
and threshold selection; test and adversarial test remain locked until after selection.


In [2]:
manifest = json.loads(MANIFEST_PATH.read_text())
PHASE1_FINGERPRINT = manifest['config_fingerprint']
SPLIT_PATHS = {
    name: DATA_DIR / f'{name}.parquet'
    for name in ('train', 'val', 'test', 'adversarial_test')
}
frames = {name: pd.read_parquet(path) for name, path in SPLIT_PATHS.items()}
for name, frame in frames.items():
    expected = manifest['counts'][name]
    assert len(frame) == expected['rows']
    assert int(frame['label'].sum()) == expected['phish']
    assert {'subject', 'body', 'label', 'augmented'}.issubset(frame.columns)
assert not frames['val']['augmented'].any()
assert not frames['test']['augmented'].any()
assert frames['adversarial_test']['augmented'].all()

def email_text(frame):
    subject = frame['subject'].fillna('').astype(str)
    body = frame['body'].fillna('').astype(str)
    return ('subject ' + subject + ' body ' + body).tolist()

texts = {name: email_text(frame) for name, frame in frames.items()}
labels = {name: frames[name]['label'].to_numpy(dtype=np.float32) for name in frames}
print({name: len(values) for name, values in texts.items()})

{'train': 137881, 'val': 16116, 'test': 16116, 'adversarial_test': 7465}


## 2. Train-Only Vocabulary and Tokenized Datasets

In [3]:
TEXT_CONFIG = {
    'max_vocab': 50000,
    'min_token_frequency': 2,
    'max_tokens': 384,
    'embedding_dim': 128,
    'hidden_dim': 128,
    'cnn_channels': 128,
    'dropout': 0.25,
    'batch_size': 512,
    'learning_rate': 2e-3,
    'weight_decay': 1e-4,
    'max_epochs': 20,
    'patience': 4,
}
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:[._@:/-][a-z0-9]+)*", re.IGNORECASE)
PAD_TOKEN, UNK_TOKEN = '<pad>', '<unk>'

def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())[:TEXT_CONFIG['max_tokens']]

token_counts = Counter(token for text in texts['train'] for token in tokenize(text))
vocab_tokens = [
    token for token, count in token_counts.most_common(TEXT_CONFIG['max_vocab'] - 2)
    if count >= TEXT_CONFIG['min_token_frequency']
]
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1, **{token: index + 2 for index, token in enumerate(vocab_tokens)}}
assert len(vocab) <= TEXT_CONFIG['max_vocab']

def encode(text):
    encoded = [vocab.get(token, vocab[UNK_TOKEN]) for token in tokenize(text)]
    return torch.tensor(encoded or [vocab[UNK_TOKEN]], dtype=torch.long)

class EmailTextDataset(Dataset):
    def __init__(self, text_values, target_values):
        self.sequences = [encode(text) for text in text_values]
        self.targets = torch.tensor(target_values, dtype=torch.float32)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        return self.sequences[index], self.targets[index]

def collate(batch):
    sequences, targets = zip(*batch)
    return pad_sequence(sequences, batch_first=True, padding_value=vocab[PAD_TOKEN]), torch.stack(targets)

datasets = {name: EmailTextDataset(texts[name], labels[name]) for name in texts}
print({'vocab_size': len(vocab), 'train_examples': len(datasets['train'])})

{'vocab_size': 50000, 'train_examples': 137881}


## 3. Train CNN + Bidirectional GRU

In [4]:
class TextCnnBiGru(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config['embedding_dim'], padding_idx=vocab[PAD_TOKEN])
        self.cnn = nn.Conv1d(config['embedding_dim'], config['cnn_channels'], kernel_size=3, padding=1)
        self.gru = nn.GRU(config['cnn_channels'], config['hidden_dim'], batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Dropout(config['dropout']),
            nn.Linear(config['hidden_dim'] * 2, config['hidden_dim']),
            nn.ReLU(),
            nn.Dropout(config['dropout']),
            nn.Linear(config['hidden_dim'], 1),
        )

    def forward(self, token_ids):
        mask = token_ids.ne(vocab[PAD_TOKEN])
        embedded = self.embedding(token_ids).transpose(1, 2)
        features = torch.relu(self.cnn(embedded)).transpose(1, 2)
        sequence, _ = self.gru(features)
        masked = sequence.masked_fill(~mask.unsqueeze(-1), float('-inf'))
        pooled = masked.max(dim=1).values
        return self.head(pooled).squeeze(1)

def predict(model, dataset, device, use_bf16):
    loader = DataLoader(dataset, batch_size=TEXT_CONFIG['batch_size'], shuffle=False, collate_fn=collate, pin_memory=False)
    probabilities = []
    model.eval()
    with torch.inference_mode():
        for token_ids, _ in loader:
            token_ids = token_ids.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
                logits = model(token_ids)
            probabilities.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(probabilities)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Phase 3 is intended to train in the GPU notebook container.'
use_bf16 = torch.cuda.is_bf16_supported()
model = TextCnnBiGru(len(vocab), TEXT_CONFIG).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=TEXT_CONFIG['learning_rate'], weight_decay=TEXT_CONFIG['weight_decay'])
positive_weight = float((labels['train'] == 0).sum() / (labels['train'] == 1).sum())
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=device))
train_loader = DataLoader(datasets['train'], batch_size=TEXT_CONFIG['batch_size'], shuffle=True, generator=torch.Generator().manual_seed(SEED), collate_fn=collate, pin_memory=False)

best_loss, best_epoch, stale_epochs, best_state = float('inf'), 0, 0, None
history = []
for epoch in range(1, TEXT_CONFIG['max_epochs'] + 1):
    model.train()
    total_loss = 0.0
    for token_ids, targets in train_loader:
        token_ids, targets = token_ids.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
            loss = criterion(model(token_ids), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(targets)
    val_probabilities = predict(model, datasets['val'], device, use_bf16)
    validation_loss = log_loss(labels['val'], val_probabilities, labels=[0, 1])
    history.append({'epoch': epoch, 'train_loss': total_loss / len(datasets['train']), 'validation_log_loss': validation_loss})
    print(history[-1])
    if validation_loss < best_loss - 1e-5:
        best_loss, best_epoch, stale_epochs = validation_loss, epoch, 0
        best_state = deepcopy({key: value.detach().cpu() for key, value in model.state_dict().items()})
    else:
        stale_epochs += 1
    torch.cuda.empty_cache()
    gc.collect()
    if stale_epochs >= TEXT_CONFIG['patience']:
        break

assert best_state is not None
model.load_state_dict(best_state)
model.to(device)
print({'device': str(device), 'bf16': use_bf16, 'best_epoch': best_epoch, 'best_validation_log_loss': best_loss})

Traceback (most recent call last):
  File "/tmp/claude-1000/-home-spark-projects/42f37dc1-0f43-4acb-9741-a0502f43b46b/scratchpad/run_nb.py", line 31, in <module>
    exec(compile(code, f'cell{i}', 'exec'), ns)
  File "cell7", line 39, in <module>
  File "/home/spark/projects/data_cleaning_generic/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1384, in to
    return self._apply(convert)
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/spark/projects/data_cleaning_generic/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 934, in _apply
    module._apply(fn)
  File "/home/spark/projects/data_cleaning_generic/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 965, in _apply
    param_applied = fn(param)
                    ^^^^^^^^^
  File "/home/spark/projects/data_cleaning_generic/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1370, in convert
    return t.to(
           ^^^^^
torch.AcceleratorError: CUDA error

## 4. Select on Validation, Then Evaluate Locked Holdouts

In [ ]:
def metrics(targets, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(targets, predictions, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold), 'precision': float(precision_score(targets, predictions, zero_division=0)),
        'recall': float(recall_score(targets, predictions, zero_division=0)), 'f1': float(f1_score(targets, predictions, zero_division=0)),
        'roc_auc': float(roc_auc_score(targets, probabilities)), 'average_precision': float(average_precision_score(targets, probabilities)),
        'brier': float(brier_score_loss(targets, probabilities)), 'log_loss': float(log_loss(targets, probabilities, labels=[0, 1])),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp), 'false_positive_rate': float(fp / (fp + tn)),
    }

val_probabilities = predict(model, datasets['val'], device, use_bf16)
threshold_grid = np.linspace(0.01, 0.99, 99)
threshold = max(threshold_grid, key=lambda value: metrics(labels['val'], val_probabilities, value)['f1'])
validation_metrics = metrics(labels['val'], val_probabilities, threshold)
test_probabilities = predict(model, datasets['test'], device, use_bf16)
adversarial_probabilities = predict(model, datasets['adversarial_test'], device, use_bf16)
test_metrics = metrics(labels['test'], test_probabilities, threshold)
adversarial_metrics = {
    'threshold': float(threshold), 'phishing_recall': float((adversarial_probabilities >= threshold).mean()),
    'probability_mean': float(adversarial_probabilities.mean()), 'probability_median': float(np.median(adversarial_probabilities)),
}
print('VALIDATION', validation_metrics)
print('LOCKED REAL TEST', test_metrics)
print('LOCKED ADVERSARIAL TEST', adversarial_metrics)

## 5. Save Phase 3 Experiment Artifacts

In [ ]:
run_id = f'phase3-cnn-bigru-{DATASET_VERSION}-{PHASE1_FINGERPRINT[:16]}'
output_dir = PROJECT_DIR / 'models' / run_id
output_dir.mkdir(parents=True, exist_ok=False)
torch.save(best_state, output_dir / 'model_state.pt')
(output_dir / 'vocab.json').write_text(json.dumps(vocab, sort_keys=True))
artifacts = {
    'run_id': run_id, 'created_utc': datetime.now(timezone.utc).isoformat(),
    'dataset_version': DATASET_VERSION,
    'phase1_fingerprint': PHASE1_FINGERPRINT, 'architecture': 'CNN + bidirectional GRU',
    'text_config': TEXT_CONFIG, 'best_epoch': best_epoch, 'validation': validation_metrics,
    'test': test_metrics, 'adversarial_test': adversarial_metrics,
    'dataset_counts': manifest['counts'],
    'subject_twin_rate': manifest.get('subject_twin_rate'),
}
(output_dir / 'metrics.json').write_text(json.dumps(artifacts, indent=2, sort_keys=True))
(output_dir / 'training_history.json').write_text(json.dumps(history, indent=2))
print(output_dir)
print(json.dumps({'validation': validation_metrics, 'test': test_metrics,
                  'adversarial_test': adversarial_metrics}, indent=2, sort_keys=True))
